## H1: Room Type vs. Price

**H0:** There is no difference in mean price between entire-home/apt listings 
and private-room listings (μ_entire = μ_private)

**H1:** Entire-home/apt listings have significantly higher mean price than 
private-room listings (μ_entire > μ_private)

**Test:** Independent samples t-test (one-tailed), assumption-dependent — 
Mann-Whitney U as fallback if normality/variance assumptions fail

**Effect size:** Cohen's d

In [3]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from scipy import stats
from src.utils.db import get_engine

engine = get_engine()

schema_query = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'dim_listing'
ORDER BY ordinal_position;
"""
print(pd.read_sql(schema_query, engine))

          column_name         data_type
0         listing_key            bigint
1             host_id            bigint
2           host_name              text
3   host_is_superhost              text
4   host_tenure_years            bigint
5       property_type              text
6           room_type              text
7        accommodates            bigint
8            bedrooms  double precision
9                beds  double precision
10         price_base  double precision
11     rating_overall  double precision
12   neighbourhood_id            bigint
13          has_price           boolean


In [4]:
distinct_query = """
SELECT room_type, COUNT(*) AS n
FROM dim_listing
GROUP BY room_type
ORDER BY n DESC;
"""
print(pd.read_sql(distinct_query, engine))

         room_type     n
0  entire home/apt  3827
1     private room  1392
2      shared room    18
3       hotel room    12


Pull data

In [5]:
h1_query = """
SELECT price_base, room_type
FROM dim_listing
WHERE has_price = true
  AND room_type IN ('entire home/apt', 'private room');
"""
h1_df = pd.read_sql(h1_query, engine)

entire = h1_df[h1_df['room_type'] == 'entire home/apt']['price_base']
private = h1_df[h1_df['room_type'] == 'private room']['price_base']

print(f"Entire home/apt: n={len(entire)}, mean={entire.mean():.2f}, median={entire.median():.2f}")
print(f"Private room:    n={len(private)}, mean={private.mean():.2f}, median={private.median():.2f}")

Entire home/apt: n=2574, mean=248.33, median=190.03
Private room:    n=929, mean=104.06, median=87.00


In [6]:
# Normality check — D'Agostino's K^2 test (better than Shapiro for n > 5000, works fine here too)
stat_entire, p_entire = stats.normaltest(entire)
stat_private, p_private = stats.normaltest(private)

print("Normality (D'Agostino K^2):")
print(f"  Entire home/apt: statistic={stat_entire:.2f}, p={p_entire:.6f}")
print(f"  Private room:    statistic={stat_private:.2f}, p={p_private:.6f}")

# Variance homogeneity — Levene's test
levene_stat, levene_p = stats.levene(entire, private)
print(f"\nLevene's test for equal variances: statistic={levene_stat:.2f}, p={levene_p:.6f}")

Normality (D'Agostino K^2):
  Entire home/apt: statistic=5766.35, p=0.000000
  Private room:    statistic=1192.28, p=0.000000

Levene's test for equal variances: statistic=46.88, p=0.000000


### Mann-Whitney U test(Since both assumptions are violated, we don't use the t-test)

In [7]:
# Mann-Whitney U test (one-tailed: testing if entire-home is stochastically greater)
u_stat, p_value = stats.mannwhitneyu(entire, private, alternative='greater')

print(f"Mann-Whitney U statistic: {u_stat:.2f}")
print(f"p-value: {p_value:.10f}")

# Effect size for Mann-Whitney: rank-biserial correlation
# r = 1 - (2U) / (n1 * n2)
n1, n2 = len(entire), len(private)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
print(f"\nRank-biserial correlation (effect size): {rank_biserial:.4f}")

Mann-Whitney U statistic: 1853750.00
p-value: 0.0000000000

Rank-biserial correlation (effect size): -0.5504


### H1 Interpretation

**Result:** The Mann-Whitney U test found a statistically significant difference 
in price between entire-home/apt and private-room listings (U = 1,853,750, 
p < 0.0001). We reject the null hypothesis.

**Effect size:** Rank-biserial correlation = 0.55 (magnitude), indicating a 
**large effect** — entire-home/apt listings are very consistently priced higher 
than private-room listings, not just on average but across the distribution.

**Business interpretation:** Entire-home/apt listings command a substantial and 
statistically robust price premium over private rooms in the Vaud market 
(median 190 CHF vs. 87 CHF — roughly 2.2x). This isn't a marginal statistical 
artifact from a few outliers; the large effect size confirms it holds broadly 
across the distribution. For hosts, converting a private-room listing to a 
full-unit rental (where feasible) represents one of the clearest, most 
data-backed levers for revenue growth in this market. For investors evaluating 
Vaud properties, entire-unit configurations offer meaningfully higher revenue 
ceiling, though — as established in EDA — also wider price variance and 
therefore higher risk/reward.

## H2: Superhost Status vs. Review Score

**H0:** There is no difference in mean review score between superhost and 
non-superhost listings (μ_super = μ_non-super)

**H1:** Superhost listings have significantly higher mean review scores than 
non-superhost listings (μ_super > μ_non-super)

**Test:** Independent samples t-test (one-tailed), pending assumption checks — 
Mann-Whitney U fallback if assumptions violated

**Effect size:** Cohen's d (or rank-biserial correlation if non-parametric)

### Pull the Data

In [9]:
h2_query = """
SELECT rating_overall, host_is_superhost
FROM dim_listing
WHERE rating_overall IS NOT NULL
  AND host_is_superhost IN ('t', 'f');
"""
h2_df = pd.read_sql(h2_query, engine)

superhost = h2_df[h2_df['host_is_superhost'] == 't']['rating_overall']
non_superhost = h2_df[h2_df['host_is_superhost'] == 'f']['rating_overall']

print(f"Superhost:     n={len(superhost)}, mean={superhost.mean():.4f}, median={superhost.median():.4f}")
print(f"Non-superhost: n={len(non_superhost)}, mean={non_superhost.mean():.4f}, median={non_superhost.median():.4f}")

Superhost:     n=1076, mean=4.8568, median=4.9000
Non-superhost: n=3034, mean=4.7419, median=4.8800


### Assumption checks

In [10]:
# Normality check
stat_super, p_super = stats.normaltest(superhost)
stat_nonsuper, p_nonsuper = stats.normaltest(non_superhost)

print("Normality (D'Agostino K^2):")
print(f"  Superhost:     statistic={stat_super:.2f}, p={p_super:.6f}")
print(f"  Non-superhost: statistic={stat_nonsuper:.2f}, p={p_nonsuper:.6f}")

# Variance homogeneity
levene_stat, levene_p = stats.levene(superhost, non_superhost)
print(f"\nLevene's test for equal variances: statistic={levene_stat:.2f}, p={levene_p:.6f}")

Normality (D'Agostino K^2):
  Superhost:     statistic=693.95, p=0.000000
  Non-superhost: statistic=2540.43, p=0.000000

Levene's test for equal variances: statistic=119.96, p=0.000000


### Mann-Whitney U + effect size

In [11]:
u_stat, p_value = stats.mannwhitneyu(superhost, non_superhost, alternative='greater')

print(f"Mann-Whitney U statistic: {u_stat:.2f}")
print(f"p-value: {p_value:.10f}")

n1, n2 = len(superhost), len(non_superhost)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
print(f"\nRank-biserial correlation (effect size): {rank_biserial:.4f}")

Mann-Whitney U statistic: 1750654.50
p-value: 0.0001556843

Rank-biserial correlation (effect size): -0.0725


### H2 Interpretation

**Result:** The Mann-Whitney U test found a statistically significant difference 
in review scores between superhost and non-superhost listings (U = 1,750,654.50, 
p = 0.000156). We reject the null hypothesis.

**Effect size:** Rank-biserial correlation = 0.07 (magnitude) — a **very small 
effect**, despite statistical significance.

**Business interpretation:** While superhosts do score significantly higher on 
reviews (median 4.90 vs. 4.88), the practical gap is minimal — both groups 
cluster in the same "excellent" range. This is partly explained by the 
superhost qualification criteria itself, which requires a 4.8+ rating as a 
prerequisite for the status, so some of this relationship is structural rather 
than purely causal. The larger takeaway from earlier host analysis stands: 
superhost status appears to function more as a **booking/visibility advantage** 
(driven by Airbnb's algorithm and badge visibility) than as a meaningful rating 
differentiator. Hosts chasing superhost status for the review-score boost alone 
would be optimizing for a marginal gain; the real value lies elsewhere.

## H3: Review Count vs. Price

**H0:** There is no difference in mean price between listings with more than 
10 reviews and listings with 10 or fewer reviews (μ_high = μ_low)

**H1:** Listings with more than 10 reviews have significantly different mean 
price than listings with 10 or fewer reviews (μ_high ≠ μ_low)

**Test:** Independent samples t-test (two-tailed), pending assumption checks — 
Mann-Whitney U fallback if assumptions violated

**Effect size:** Cohen's d (or rank-biserial correlation if non-parametric)

### Pull the data

In [13]:
h3_query = """
SELECT 
    dl.price_base,
    COALESCE(rc.review_count, 0) AS review_count
FROM dim_listing dl
LEFT JOIN (
    SELECT listing_key, COUNT(*) AS review_count
    FROM fact_reviews
    GROUP BY listing_key
) rc ON dl.listing_key = rc.listing_key
WHERE dl.has_price = true;
"""
h3_df = pd.read_sql(h3_query, engine)

high_reviews = h3_df[h3_df['review_count'] > 10]['price_base']
low_reviews = h3_df[h3_df['review_count'] <= 10]['price_base']

print(f">10 reviews:  n={len(high_reviews)}, mean={high_reviews.mean():.2f}, median={high_reviews.median():.2f}")
print(f"≤10 reviews:  n={len(low_reviews)}, mean={low_reviews.mean():.2f}, median={low_reviews.median():.2f}")

>10 reviews:  n=1569, mean=169.98, median=142.50
≤10 reviews:  n=1958, mean=240.89, median=172.42


### Assumption Checks

In [14]:
stat_high, p_high = stats.normaltest(high_reviews)
stat_low, p_low = stats.normaltest(low_reviews)

print("Normality (D'Agostino K^2):")
print(f"  >10 reviews: statistic={stat_high:.2f}, p={p_high:.6f}")
print(f"  ≤10 reviews: statistic={stat_low:.2f}, p={p_low:.6f}")

levene_stat, levene_p = stats.levene(high_reviews, low_reviews)
print(f"\nLevene's test for equal variances: statistic={levene_stat:.2f}, p={levene_p:.6f}")

Normality (D'Agostino K^2):
  >10 reviews: statistic=851.62, p=0.000000
  ≤10 reviews: statistic=4246.11, p=0.000000

Levene's test for equal variances: statistic=22.53, p=0.000002


### Mann-Whitney U + effect size

In [15]:
u_stat, p_value = stats.mannwhitneyu(high_reviews, low_reviews, alternative='two-sided')

print(f"Mann-Whitney U statistic: {u_stat:.2f}")
print(f"p-value: {p_value:.10f}")

n1, n2 = len(high_reviews), len(low_reviews)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
print(f"\nRank-biserial correlation (effect size): {rank_biserial:.4f}")

Mann-Whitney U statistic: 1310592.50
p-value: 0.0000000000

Rank-biserial correlation (effect size): 0.1468


### H3 Interpretation

**Result:** The Mann-Whitney U test found a statistically significant difference 
in price between listings with more than 10 reviews and listings with 10 or 
fewer reviews (U = 1,310,592.50, p < 0.0001). We reject the null hypothesis.

**Effect size:** Rank-biserial correlation = 0.15 (magnitude) — a **small 
effect**, closer to negligible than to moderate.

**Business interpretation:** Contrary to a naive assumption that higher review 
counts might track with higher prices (as a proxy for "popularity" or 
"established" listings), the direction here actually runs the opposite way — 
listings with ≤10 reviews have a *higher* median price (172 CHF) than listings 
with >10 reviews (142 CHF). This is consistent with earlier EDA findings that 
review frequency does not track with price, and that entire-home/apt listings 
(which skew toward higher prices) tend to have longer stays and thus 
accumulate reviews more slowly than lower-priced private/shared rooms with 
higher guest turnover. The effect, while statistically real, is modest in 
practical terms — review count alone is a weak predictor of price and 
shouldn't be over-relied upon as a market signal.

## H4: Neighbourhood vs. Price

**H0:** Mean rank of price is equal across all neighbourhoods (no neighbourhood 
effect on price)

**H1:** At least one neighbourhood's price distribution differs significantly 
from the others

**Test:** One-way ANOVA was the first choice, but normality (D'Agostino K², 
p < 0.0001) and variance homogeneity (Levene's, p < 0.0001) were both violated 
across the 97 qualifying neighbourhoods — so **Kruskal-Wallis H-test** was used 
instead.

**Effect size:** Epsilon-squared (ε²) — the non-parametric counterpart to 
eta-squared

### Pull data joined to neighbourhood name

In [18]:
h4_query = """
SELECT dl.price_base, dn.neighbourhood_name
FROM dim_listing dl
JOIN dim_neighbourhood dn ON dl.neighbourhood_id = dn.neighbourhood_id
WHERE dl.has_price = true;
"""
h4_df = pd.read_sql(h4_query, engine)

print(f"Total rows pulled: {len(h4_df)}")
h4_df.head()

Total rows pulled: 3527


,price_base,neighbourhood_name
0,139.00,Ormont-Dessus
1,813.25,Mont-Sur-Rolle
2,420.00,Ollon
3,223.33,Lausanne
4,239.00,Chardonne


In [19]:
group_sizes = h4_df.groupby('neighbourhood_name').size().sort_values(ascending=False)

print(f"Total neighbourhoods with priced listings: {len(group_sizes)}")
print(f"\nGroup size distribution:")
print(group_sizes.describe())
print(f"\nNeighbourhoods with fewer than 5 listings: {(group_sizes < 5).sum()}")

Total neighbourhoods with priced listings: 228

Group size distribution:
count    228.000000
mean      15.469298
std       55.269575
min        1.000000
25%        1.000000
50%        3.000000
75%       10.000000
max      678.000000
dtype: float64

Neighbourhoods with fewer than 5 listings: 131


### Filter and Finalize the H4 dataset

**Note on H4 scope:** Of 228 neighbourhoods with priced listings, 131 (57%) had 
fewer than 5 listings — too small a sample to estimate reliable within-group 
variance. These were excluded prior to testing, retaining neighbourhoods with 
n ≥ 5 listings. This is a standard practice for group-comparison tests and is 
documented here as an explicit assumption, not a silent exclusion.

In [20]:
MIN_LISTINGS = 5

valid_neighbourhoods = group_sizes[group_sizes >= MIN_LISTINGS].index
h4_filtered = h4_df[h4_df['neighbourhood_name'].isin(valid_neighbourhoods)]

print(f"Neighbourhoods before filter: {len(group_sizes)}")
print(f"Neighbourhoods after filter (>= {MIN_LISTINGS} listings): {len(valid_neighbourhoods)}")
print(f"Listings before filter: {len(h4_df)}")
print(f"Listings after filter: {len(h4_filtered)}")

Neighbourhoods before filter: 228
Neighbourhoods after filter (>= 5 listings): 97
Listings before filter: 3527
Listings after filter: 3288


### Assumption Checks

In [21]:
# Prepare list of price arrays, one per neighbourhood
groups = [group['price_base'].values for name, group in h4_filtered.groupby('neighbourhood_name')]

# Overall normality check (pooled residuals approach — check normality of price 
# within the filtered dataset as a practical proxy, since checking 97 groups 
# individually isn't meaningful with such variable sample sizes)
overall_stat, overall_p = stats.normaltest(h4_filtered['price_base'])
print(f"Normality (D'Agostino K^2) on filtered price_base: statistic={overall_stat:.2f}, p={overall_p:.6f}")

# Levene's test across all 97 neighbourhood groups simultaneously
levene_stat, levene_p = stats.levene(*groups)
print(f"\nLevene's test for equal variances across {len(groups)} groups: statistic={levene_stat:.2f}, p={levene_p:.6f}")

Normality (D'Agostino K^2) on filtered price_base: statistic=6110.57, p=0.000000

Levene's test for equal variances across 97 groups: statistic=4.95, p=0.000000


### Kruskal-Wallis test + effect size

In [23]:
h_stat, p_value = stats.kruskal(*groups)

print(f"Kruskal-Wallis H statistic: {h_stat:.2f}")
print(f"p-value: {p_value:.10f}")

# Effect size for Kruskal-Wallis: epsilon-squared (analogous to eta-squared for ANOVA)
n_total = len(h4_filtered)
k_groups = len(groups)
epsilon_squared = (h_stat - k_groups + 1) / (n_total - k_groups)
print(f"\nEpsilon-squared (effect size): {epsilon_squared:.4f}")

Kruskal-Wallis H statistic: 662.88
p-value: 0.0000000000

Epsilon-squared (effect size): 0.1776


### H4 Interpretation

**Result:** The Kruskal-Wallis H-test found a statistically significant 
difference in price across neighbourhoods (H = 662.88, p < 0.0001, 
df = 96). We reject the null hypothesis. Analysis was restricted to 97 
neighbourhoods with at least 5 priced listings (3,288 of 3,527 total 
listings), excluding 131 neighbourhoods too small to estimate reliable 
within-group variance.

**Effect size:** Epsilon-squared = 0.18 — a **large effect**, the strongest 
of all hypotheses tested in this section.

**Business interpretation:** Neighbourhood is a powerful, statistically robust 
driver of price in the Vaud Airbnb market — far more so than review count 
(H3, small effect) or superhost status (H2, negligible effect). This 
reinforces the EDA finding that smaller resort-adjacent neighbourhoods 
(Ormont-Dessus, Gryon, Ollon) command substantially higher prices than 
high-volume hubs like Lausanne. For hosts and investors, **location is the 
single strongest lever identified across this entire statistical analysis** — 
more predictive of price than host reputation, review volume, or property 
turnover. Market entry or expansion decisions should weight neighbourhood 
choice heavily above other factors tested here.

## H5: Weekend vs. Weekday Pricing

**H0:** There is no difference in price between weekend and weekday listings

**H1:** Weekend and weekday prices differ significantly

**Status: Not testable with available data.**

`fact_calendar` tracks daily availability (`is_available`), minimum/maximum 
night requirements, and date, but **does not include a price field**. This is 
a source data limitation confirmed during the transformation and modeling 
phases (Inside Airbnb's Vaud export does not provide date-varying/dynamic 
pricing — only a single static `price_base` per listing in `dim_listing`). 
True weekend-vs-weekday price variation cannot be derived from this dataset, 
since no table captures how price changes by date.

**Decision:** H5 is documented here as untestable rather than forced through 
a proxy metric that wouldn't actually answer the stated hypothesis. This is 
consistent with the "document and move on" approach applied throughout this 
project for confirmed source-data gaps (e.g., the ~33% null price rate, the 
absent host response-rate field).

**Reframing considered:** An earlier EDA finding (Section 4.3) showed minimum-
night requirements are flat across all months (median 2 nights, 75th 
percentile ~4 nights) with no seasonal variation — this could be tested 
statistically as a substitute seasonal-pricing-behavior hypothesis, but it 
answers a different question than H5 as originally specified, and is 
therefore treated as a separate, optional extension rather than a direct 
replacement.

# 5.2 Confidence Intervals & Effect Sizes

**Methodology note:** Since Section 5.1 showed that price is not normally distributed across all groups tested (room type, review count, 
neighbourhood), confidence intervals here use **bootstrap resampling** 
(10,000 resamples, percentile method) rather than standard parametric 
(t-distribution) CIs. This keeps the methodology consistent with the 
non-parametric tests already applied and avoids assuming normality the 
data has repeatedly failed to satisfy.

## Bootstrap CI function

In [25]:
def bootstrap_ci(data, n_boot=10000, ci=95, seed=42):
    """Bootstrap confidence interval for the mean."""
    rng = np.random.default_rng(seed)
    data = np.asarray(data)
    boot_means = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(data, size=len(data), replace=True)
        boot_means[i] = sample.mean()
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return data.mean(), lower, upper

## Confidence Interval by room type:

In [28]:
room_type_query = """
SELECT price_base, room_type
FROM dim_listing
WHERE has_price = true;
"""
room_df = pd.read_sql(room_type_query, engine)

room_type_cis = []

for rtype, group in room_df.groupby('room_type'):
    prices = group['price_base'].values
    mean_p, lower, upper = bootstrap_ci(prices, n_boot=10000)
    room_type_cis.append({
        'room_type': rtype,
        'n': len(prices),
        'mean_price': mean_p,
        'ci_lower': lower,
        'ci_upper': upper,
        'ci_width': upper - lower
    })

room_type_ci_df = pd.DataFrame(room_type_cis).sort_values('mean_price', ascending=False)
room_type_ci_df

,room_type,n,mean_price,ci_lower,ci_upper,ci_width
0,entire home/apt,2574,248.327327,234.165415,265.489008,31.323593
1,hotel room,8,131.426250,66.755000,192.500000,125.745000
2,private room,929,104.059882,97.871285,110.812299,12.941014
3,shared room,16,90.763125,47.240094,146.100047,98.859953




95% bootstrap CIs (10,000 resamples) for mean price by room type:

| Room Type | n | Mean Price (CHF) | 95% CI | CI Width |
|---|---|---|---|---|
| Entire home/apt | 2,574 | 248.33 | [234.17, 265.49] | 31.32 |
| Hotel room | 8 | 131.43 | [66.76, 192.50] | 125.75 |
| Private room | 929 | 104.06 | [97.87, 110.81] | 12.94 |
| Shared room | 16 | 90.76 | [47.24, 146.10] | 98.86 |

**Interpretation:** Entire-home and private-room estimates are precise, given 
large sample sizes — their CIs are narrow and don't overlap, reinforcing H1's 
conclusion with a concrete price range rather than just a significant p-value. 
Hotel room and shared room estimates should be treated with caution: their 
sample sizes (n=8 and n=16 respectively) are too small to produce a stable 
mean estimate, reflected in CI widths 4-10x wider than the two dominant 
categories. These two segments are a negligible share of the Vaud market 
(24 of 3,527 priced listings, <1%) and any business conclusions drawn from 
them should be treated as directional at best, not statistically reliable.

## Bootstrap Confidence Interval by neighbourhood 

In [29]:
neighbourhood_cis = []

for name, group in h4_filtered.groupby('neighbourhood_name'):
    prices = group['price_base'].values
    mean_p, lower, upper = bootstrap_ci(prices, n_boot=2000)  # fewer resamples per group, 97 groups total
    neighbourhood_cis.append({
        'neighbourhood_name': name,
        'n': len(prices),
        'mean_price': mean_p,
        'ci_lower': lower,
        'ci_upper': upper,
        'ci_width': upper - lower
    })

neighbourhood_ci_df = pd.DataFrame(neighbourhood_cis).sort_values('mean_price', ascending=False)
neighbourhood_ci_df.head(10)

,neighbourhood_name,n,mean_price,ci_lower,ci_upper,ci_width
24,Crans-Près-Céligny,6,1698.208333,169.708333,4680.500000,4510.791667
52,Mies,8,640.608750,221.457281,1337.843125,1116.385844
77,Rougemont,44,459.628864,288.319244,654.178693,365.859449
58,Noville,8,449.595000,234.890312,710.663563,475.773250
49,Lonay,5,388.750000,160.284000,642.804000,482.520000
80,Saint-Légier-La Chiésaz,11,384.372727,219.315955,566.602182,347.286227
21,Coppet,8,354.870000,207.571656,502.231719,294.660062
19,Château-D'Oex,96,352.638438,235.125370,502.019461,266.894091
5,Belmont-Sur-Lausanne,12,326.362500,172.350750,482.594167,310.243417
33,Faoug,5,315.234000,247.000000,389.302000,142.302000




95% bootstrap CIs (2,000 resamples per group) computed across the 97 
neighbourhoods with n ≥ 5 priced listings (same filter established in H4).

**Top 10 by mean price:**

| Neighbourhood | n | Mean Price | 95% CI | CI Width |
|---|---|---|---|---|
| Crans-Près-Céligny | 6 | 1698.21 | [169.71, 4680.50] | 4510.79 |
| Mies | 8 | 640.61 | [221.46, 1337.84] | 1116.39 |
| Rougemont | 44 | 459.63 | [288.32, 654.18] | 365.86 |
| Noville | 8 | 449.60 | [234.89, 710.66] | 475.77 |
| Lonay | 5 | 388.75 | [160.28, 642.80] | 482.52 |
| Saint-Légier-La Chiésaz | 11 | 384.37 | [219.32, 566.60] | 347.29 |
| Coppet | 8 | 354.87 | [207.57, 502.23] | 294.66 |
| Château-D'Oex | 96 | 352.64 | [235.13, 502.02] | 266.89 |
| Belmont-Sur-Lausanne | 12 | 326.36 | [172.35, 482.59] | 310.24 |
| Faoug | 5 | 315.23 | [247.00, 389.30] | 142.30 |

**Bottom 10 by mean price:**

| Neighbourhood | n | Mean Price | 95% CI | CI Width |
|---|---|---|---|---|
| Chavannes-Près-Renens | 14 | 99.42 | [62.37, 147.06] | 84.69 |
| Moudon | 12 | 97.46 | [87.87, 107.08] | 19.21 |
| Rivaz | 8 | 94.18 | [42.45, 154.61] | 112.16 |
| Yverdon-Les-Bains | 25 | 92.90 | [70.74, 118.82] | 48.08 |
| Echallens | 5 | 87.26 | [35.72, 143.20] | 107.48 |
| Le Vaud | 6 | 84.60 | [29.90, 144.74] | 114.84 |
| Gland | 7 | 67.93 | [22.24, 135.01] | 112.77 |
| Assens | 6 | 56.49 | [38.64, 75.46] | 36.82 |
| Roche (Vaud) | 13 | 56.45 | [38.10, 76.94] | 38.84 |
| Orbe | 5 | 41.87 | [11.44, 72.24] | 60.80 |

**Interpretation:** Only the larger-sample neighbourhoods give reliable 
estimates — Château-D'Oex (n=96) and Yverdon-Les-Bains (n=25) both have 
tight CIs under 300 CHF wide. Most others in these tables have n=5-13 and 
CI widths several times their own mean (e.g. Crans-Près-Céligny: CI 
[169.71, 4680.50]), meaning we can't pin down their true average price 
with any precision. **So the "most/least expensive neighbourhood" rankings 
shouldn't be taken at face value — they're mostly noise from small samples.** 
The safer takeaway is what H4 already showed: neighbourhood has a real, 
large effect on price overall (ε² = 0.18), even if ranking individual small 
neighbourhoods needs more data than we currently have.

## Effect Size Summary

**Note:** All tests in 5.1 used non-parametric methods (data failed normality 
checks throughout), so effect sizes here are the non-parametric equivalents 
of Cohen's d and eta-squared: rank-biserial correlation (r) for two-group 
comparisons, and epsilon-squared (ε²) for the multi-group comparison.

| Hypothesis | Test | p-value | Effect Size | Magnitude |
|---|---|---|---|---|
| H1: Room type vs. price | Mann-Whitney U | < 0.0001 | r = 0.55 | Large |
| H2: Superhost vs. review score | Mann-Whitney U | 0.0002 | r = 0.07 | Negligible |
| H3: Review count vs. price | Mann-Whitney U | < 0.0001 | r = 0.15 | Small |
| H4: Neighbourhood vs. price | Kruskal-Wallis | < 0.0001 | ε² = 0.18 | Large |

## Practical vs. Statistical Significance

All four hypotheses were statistically significant — but significance alone 
doesn't mean a finding matters in practice. With ~4,000 listings, even tiny, 
trivial differences become detectable (H2 is the clearest example: p = 0.0002 
looks impressive, but the effect size of 0.07 says the actual difference 
between superhost and non-superhost ratings is barely more than a coin flip).

Ranking hypotheses by what actually matters for the business, not just by 
p-value:

1. **H1 (room type) and H4 (neighbourhood)** — large effects. These are the 
   two levers that genuinely move price in this market.
2. **H3 (review count)** — small effect. Real, but not something to build a 
   pricing strategy around.
3. **H2 (superhost status)** — negligible effect. Statistically real, 
   practically irrelevant for review scores specifically (though superhost 
   status may still matter for other outcomes, like booking volume, which 
   this test didn't measure).

The lesson: p-value tells you *whether* something's happening; effect size 
tells you *whether it's worth caring about*. Reporting both, as done here, 
avoids overstating small or trivial findings just because they cleared the 
0.05 threshold.